# 03 · Experimentos y registro de modelos con MLflow

El seguimiento vincula cada pronóstico con el snapshot, los parámetros y los cortes usados para evaluarlo. El **ingenuo estacional** repite las ventas del mismo mes del año anterior. **Ridge** ajusta una regresión regularizada con un escalador entrenado dentro de cada fold y cuatro predictores históricos.

Este notebook consulta corridas, artefactos y versiones existentes. Los comandos de entrenamiento y promoción viven en el [README](../README.md).

## Conexión al servidor local

Si el servidor está apagado, ejecútalo desde la raíz en otra terminal y vuelve a correr el notebook:

```text
uv run --locked mlflow server --backend-store-uri sqlite:///mlflow.db --serve-artifacts --artifacts-destination ./mlartifacts --host 127.0.0.1 --port 5000 --workers 1
```

Por defecto se consulta `http://127.0.0.1:5000`. Puedes ajustar `MLFLOW_TRACKING_URI` antes de iniciar el kernel. Un servidor nuevo puede estar vacío. Consulta el README para generar corridas deliberadamente fuera del notebook.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.error import URLError
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from superstore_forecasting.config import (
    DATASET,
    EVAL_END,
    EVAL_START,
    RIDGE_ALPHAS,
    SHA256,
    TUNING_END,
    TUNING_START,
)
from superstore_forecasting.experiments import (
    EXPERIMENT,
    REGISTERED_MODEL,
    TRACKING_URI,
)
from superstore_forecasting.model import BASELINE, CANDIDATE

plt.style.use("seaborn-v0_8-whitegrid")
uri = os.environ.get("MLFLOW_TRACKING_URI", TRACKING_URI).rstrip("/")
client = None
try:
    with urlopen(f"{uri}/health", timeout=3) as response:
        if response.status == 200:
            client = MlflowClient(tracking_uri=uri, registry_uri=uri)
except (OSError, ValueError, URLError):
    display(
        Markdown(
            "**MLflow no está accesible.** Inicia el servidor con el comando anterior y vuelve a ejecutar."
        )
    )

## Elegir una comparación reproducible

Se elige el lote completo más reciente del snapshot verificado. Debe contener las cuatro corridas de ajuste de 2017 y los dos modelos del holdout de 2018. Los intentos parciales se excluyen. Cada lote puede identificarse por su etiqueta `comparison_id`.

In [ ]:
runs = []
batch = None
experiment = client.get_experiment_by_name(EXPERIMENT) if client else None
if experiment:
    token = None
    while True:
        page = client.search_runs(
            [experiment.experiment_id],
            filter_string=f"attributes.status = 'FINISHED' AND params.dataset_sha256 = '{SHA256}' AND params.dataset = '{DATASET}'",
            order_by=["attributes.start_time DESC"],
            page_token=token,
        )
        runs.extend(page)
        token = page.token
        if not token:
            break

In [ ]:
def run_key(run):
    phase = run.data.tags.get("phase")
    params = run.data.params
    window = (params.get("evaluation_start"), params.get("evaluation_end"))
    if phase == "tuning" and window == (TUNING_START, TUNING_END):
        if params.get("algorithm") == CANDIDATE and params.get("alpha"):
            return phase, float(params["alpha"])
    if phase == "holdout" and window == (EVAL_START, EVAL_END):
        return phase, params.get("algorithm")
    return None


expected = {("tuning", alpha) for alpha in RIDGE_ALPHAS}
expected.update({("holdout", BASELINE), ("holdout", CANDIDATE)})
batches = {}
for run in runs:
    key = run_key(run)
    comparison_id = run.data.tags.get("comparison_id")
    if comparison_id and key in expected and {"mae", "rmse"} <= run.data.metrics.keys():
        batches.setdefault(comparison_id, {}).setdefault(key, run)

for comparison_id, candidate_batch in batches.items():
    if set(candidate_batch) == expected:
        batch = candidate_batch
        print(f"Lote seleccionado: {comparison_id}")
        break
if client and batch is None:
    display(
        Markdown(
            "**No hay un lote completo para el snapshot fijado.** Consulta la sección de experimentos del README."
        )
    )

## Trazabilidad y ajuste de Ridge en 2017

Los artefactos de cada corrida incluyen `walk_forward.csv` y `evaluation.json`. En 2017 se comparan los valores de alpha mediante folds expansivos. Se elige el menor MAE y se lleva ese valor fijo al holdout.

In [ ]:
if batch is not None:
    inventory = []
    for (phase, _), run in batch.items():
        params = run.data.params
        inventory.append(
            {
                "Fase": phase,
                "Corrida": run.info.run_id,
                "Modelo": params["algorithm"],
                "Alpha": params.get("alpha"),
                "Primer mes": params["evaluation_start"],
                "Último mes": params["evaluation_end"],
                "Folds": params["folds"],
                "Snapshot": params["dataset"],
                "SHA-256": params["dataset_sha256"],
                "Artefactos de corrida": ", ".join(
                    item.path for item in client.list_artifacts(run.info.run_id)
                ),
            }
        )
    display(pd.DataFrame(inventory))
    tuning = pd.DataFrame(
        [
            dict(alpha=alpha, **batch["tuning", alpha].data.metrics)
            for alpha in RIDGE_ALPHAS
        ]
    ).set_index("alpha")
    chosen = min(RIDGE_ALPHAS, key=lambda alpha: (tuning.at[alpha, "mae"], alpha))
    assert float(batch["holdout", CANDIDATE].data.params["alpha"]) == chosen
    display(tuning[["mae", "rmse"]].round(2))
    display(Markdown(f"Alpha seleccionado en 2017: **{chosen:g}**."))
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    for metric, color in (("mae", "#267b82"), ("rmse", "#bf6335")):
        ax.plot(tuning.index, tuning[metric], "-o", label=metric.upper(), color=color)
    ax.set(
        xscale="log",
        title="Validación temporal de Ridge en 2017",
        xlabel="Alpha (escala logarítmica)",
        ylabel="Error (unidad de Sales)",
    )
    ax.legend()
    plt.show()

## Holdout de 2018

Cada predicción se hizo con los meses disponibles hasta el corte previo. MAE resume la magnitud media del error y RMSE da más peso a errores grandes. Se comparan exactamente los mismos meses. Los CSV se descargan a una carpeta temporal y se verifican contra las métricas registradas.

In [ ]:
if batch is not None:
    folds = {}
    model_names = {BASELINE: "Ingenuo estacional", CANDIDATE: "Ridge"}
    with TemporaryDirectory() as directory:
        for model_id in model_names:
            run = batch["holdout", model_id]
            download_dir = Path(directory) / model_id
            download_dir.mkdir()
            csv = client.download_artifacts(
                run.info.run_id, "walk_forward.csv", str(download_dir)
            )
            fold = (
                pd.read_csv(csv, parse_dates=["month"]).set_index("month").sort_index()
            )
            assert fold.index.equals(
                pd.date_range(EVAL_START, EVAL_END, freq="MS", name="month")
            )
            checks = {
                "mae": mean_absolute_error(fold["actual"], fold["prediction"]),
                "rmse": root_mean_squared_error(fold["actual"], fold["prediction"]),
            }
            for metric, value in checks.items():
                assert abs(value - run.data.metrics[metric]) < 1e-6
            folds[model_id] = fold
    pd.testing.assert_series_equal(
        folds[BASELINE]["actual"], folds[CANDIDATE]["actual"]
    )
    actual = folds[BASELINE]["actual"]
    forecasts = pd.DataFrame(
        {name: folds[model_id]["prediction"] for model_id, name in model_names.items()}
    )
    metrics = pd.DataFrame(
        {
            name: batch["holdout", model_id].data.metrics
            for model_id, name in model_names.items()
        }
    ).T
    display(metrics[["mae", "rmse"]].round(2))
    winner = metrics["mae"].idxmin()
    display(Markdown(f"Menor MAE en 2018: **{winner}**."))

In [ ]:
if batch is not None:
    fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
    actual.plot(ax=ax, color="#23334d", linewidth=2.5, label="Real")
    forecasts.plot(ax=ax, marker="o", color=["#267b82", "#bf6335"])
    ax.set(
        title="Ventas reales y pronosticadas en 2018",
        ylabel="Ventas (unidad de Sales)",
        xlabel="Mes",
    )
    ax.legend()
    plt.show()

### ¿Cuándo se equivocaron los modelos?

Un error con signo positivo indica sobreestimación. El valor absoluto permite localizar los meses con mayor desviación sin mezclar la magnitud con la dirección.

In [ ]:
if batch is not None:
    errors = forecasts.sub(actual, axis="index")
    labels = errors.set_axis(errors.index.strftime("%Y-%m"))
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, layout="constrained")
    labels.plot.bar(ax=axes[0], color=["#267b82", "#bf6335"])
    labels.abs().plot.bar(ax=axes[1], color=["#267b82", "#bf6335"], legend=False)
    axes[0].axhline(0, color="#23334d", linewidth=1)
    axes[0].set(title="Error con signo", ylabel="Pronóstico − real")
    axes[1].set(
        title="Magnitud del error",
        xlabel="Mes",
        ylabel="Error absoluto (unidad de Sales)",
    )
    axes[1].tick_params(axis="x", rotation=45)
    plt.show()
    display(
        pd.DataFrame(
            {
                "Mes de mayor error": errors.abs().idxmax().dt.strftime("%Y-%m"),
                "Error absoluto": errors.abs().max().round(2),
            }
        )
    )

## Model Registry

El registro asocia artefactos a versiones numeradas. `champion` es un alias que se puede mover deliberadamente tras revisar una versión. El alias consultado corresponde al estado actual del servidor y puede apuntar a una versión de otro lote.

In [ ]:
if client:
    try:
        registered = client.get_registered_model(REGISTERED_MODEL)
    except MlflowException as error:
        if error.error_code != "RESOURCE_DOES_NOT_EXIST":
            raise
        display(
            Markdown(
                "El modelo aún no tiene una entrada en el registro de este servidor."
            )
        )
    else:
        versions = client.search_model_versions(f"name = '{REGISTERED_MODEL}'")
        display(
            pd.DataFrame(
                [
                    {
                        "Versión": item.version,
                        "Estado": item.status,
                        "Modelo": item.tags.get("algorithm"),
                        "Entrenado hasta": item.tags.get("trained_through"),
                        "Lote": item.tags.get("comparison_id"),
                    }
                    for item in versions
                ]
            )
        )
        champion = registered.aliases.get("champion")
        if champion is None:
            display(Markdown("**champion** aún no está asignado."))
        else:
            display(Markdown(f"**champion** apunta a la versión **{champion}**."))

## Conclusiones y límites

El holdout permite comparar los modelos en un año posterior al ajuste de hiperparámetros. El artefacto serializado usa el cierre de su fase para inferencia y las métricas provienen de ajustes específicos de cada fold. Hay un solo año de evaluación final y el ganador carece de una evaluación posterior independiente en estos datos. El lote y el alias que se muestran dependen del backend consultado.